# Chapter 14b — Regularization (Fixing Overfitting)

> **The big idea.** In Chapter 9 we saw a high-degree polynomial **overfit**:
> it wiggled wildly to chase noise and generalized terribly. **Regularization**
> is the cure. We add a penalty that discourages large weights, so the model
> *prefers* a smoother, simpler fit unless the data really demands complexity.

By the end you will be able to:

- recreate **overfitting** with a high-degree polynomial on noisy data;
- derive and code **L2 (ridge)** regression, including the modified normal
  equation $\theta = (X^\top X + \lambda I)^{-1} X^\top y$;
- **sweep $\lambda$** and read the train-vs-validation **U-curve**, and watch the
  fitted curve get smoother as $\lambda$ grows;
- understand **L1 (lasso)** and why it drives weights to **exactly zero**
  (automatic feature selection);
- restate the **bias–variance** trade-off in terms of $\lambda$;
- see **weight decay** as L2's effect inside a gradient-descent step
  (connecting to Chapter 12).

Run every code cell with **Shift + Enter**.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

def mse(y_true, y_hat):
    return np.mean((y_hat - y_true)**2)

## 1. Recreating overfitting

We make data from a gentle true curve plus noise, then fit a **high-degree
polynomial** — far more flexible than the data needs. As in Chapter 9, it will
hug the training points and swing wildly between them.

In [ ]:
def true_fn(x):
    return np.sin(1.2 * x) + 0.5 * x        # the smooth underlying pattern

n = 25
x = np.sort(rng.uniform(0, 5, n))
y = true_fn(x) + rng.normal(0, 0.35, n)     # noisy observations

# split into train / validation (we'll use validation to tune lambda)
idx = np.arange(n); rng.shuffle(idx)
cut = int(0.6 * n)
tr, va = idx[:cut], idx[cut:]
xtr, ytr = x[tr], y[tr]
xva, yva = x[va], y[va]
print("train size:", xtr.size, " val size:", xva.size)

In [ ]:
DEGREE = 12                                  # deliberately too flexible
grid = np.linspace(0, 5, 300)

c = np.polyfit(xtr, ytr, DEGREE)             # plain (unregularized) fit
plt.scatter(xtr, ytr, label="train", color="C0")
plt.scatter(xva, yva, label="val", color="C1", marker="s")
plt.plot(grid, np.polyval(c, grid), "k", label=f"degree {DEGREE} fit")
plt.plot(grid, true_fn(grid), "g--", alpha=0.6, label="truth")
plt.ylim(-2, 5)
plt.title("Overfitting: a degree-12 polynomial chases the noise")
plt.legend(fontsize=8); plt.grid(True)
plt.show()

print("train MSE:", round(mse(ytr, np.polyval(c, xtr)), 4))
print("val   MSE:", round(mse(yva, np.polyval(c, xva)), 4))

Tiny train error, large validation error — the signature of overfitting. Notice
the wild swings come from **huge polynomial coefficients** fighting each other.
Regularization attacks exactly that.

In [ ]:
print("largest |coefficient| in the overfit model:", round(np.max(np.abs(c)), 1))
# Overfit models tend to have enormous weights. We will penalize that.

## 2. L2 regularization (ridge regression)

To fit a polynomial as linear regression we build a **design matrix** of powers:

$$X = \begin{bmatrix} 1 & x_1 & x_1^2 & \cdots & x_1^p \\
\vdots & & & & \vdots \\ 1 & x_n & x_n^2 & \cdots & x_n^p \end{bmatrix},
\qquad f_\theta(x) = X\theta.$$

Ordinary least squares minimizes $\mathrm{MSE} = \frac{1}{n}\|X\theta - y\|^2$,
solved by the **normal equation** $\theta = (X^\top X)^{-1} X^\top y$.

**Ridge** adds an **L2 penalty** on the weights to the cost:

$$J(\theta) = \frac{1}{n}\|X\theta - y\|^2 \; + \; \lambda \|\theta\|^2,
\qquad \|\theta\|^2 = \sum_j \theta_j^2.$$

Setting $\nabla J = 0$ gives the **modified normal equation**:

$$\boxed{\;\theta = (X^\top X + \lambda I)^{-1} X^\top y\;}$$

The extra $\lambda I$ "shrinks" the weights toward zero — larger $\lambda$ means a
stronger pull, hence a smoother fit. Let's code it from scratch.

In [ ]:
def poly_design(x, degree):
    """Build the (n, degree+1) matrix of powers [1, x, x^2, ..., x^degree]."""
    return np.vander(x, degree + 1, increasing=True)

def ridge_fit(x, y, degree, lam):
    """Ridge regression via the modified normal equation."""
    X = poly_design(x, degree)
    p = X.shape[1]
    I = np.eye(p)
    I[0, 0] = 0.0                 # by convention, do NOT penalize the bias term
    theta = np.linalg.solve(X.T @ X + lam * I, X.T @ y)
    return theta

def ridge_predict(theta, x):
    return poly_design(x, len(theta) - 1) @ theta

In [ ]:
# sanity check: lambda = 0 reproduces ordinary least squares
theta0 = ridge_fit(xtr, ytr, degree=3, lam=0.0)
ref = np.polyfit(xtr, ytr, 3)[::-1]          # polyfit returns high->low power
print("ridge (lam=0):", theta0)
print("polyfit ref  :", ref)
print("match:", np.allclose(theta0, ref))

Now fit the **same degree-12** model with a few values of $\lambda$ and watch the
curve relax from wild to smooth.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, lam in zip(axes, [0.0, 1e-4, 1e-1]):
    theta = ridge_fit(xtr, ytr, DEGREE, lam)
    ax.scatter(xtr, ytr, s=20, color="C0", label="train")
    ax.scatter(xva, yva, s=25, color="C1", marker="s", label="val")
    ax.plot(grid, ridge_predict(theta, grid), "k", label=f"lam={lam}")
    ax.plot(grid, true_fn(grid), "g--", alpha=0.6, label="truth")
    ax.set_ylim(-2, 5); ax.set_title(f"lambda = {lam}")
    ax.legend(fontsize=8); ax.grid(True)
axes[0].set_ylabel("y")
plt.suptitle("Ridge: larger lambda -> smoother fit")
plt.tight_layout()
plt.show()

## 3. Sweeping $\lambda$: the regularization U-curve

In Chapter 9 we varied **model complexity** (degree) to get a U-shaped test
curve. Here the degree is **fixed** and we vary the **regularization strength**
$\lambda$ instead. The shape is the same idea, mirrored:

- **$\lambda$ too small** → no penalty → **overfitting** (low train, high val).
- **$\lambda$ too large** → weights crushed toward 0 → **underfitting** (high
  train *and* val).
- **In between** → the sweet spot, the bottom of the U.

In [ ]:
lambdas = np.logspace(-6, 1, 40)             # 1e-6 ... 1e1, log-spaced
train_errs, val_errs = [], []
for lam in lambdas:
    theta = ridge_fit(xtr, ytr, DEGREE, lam)
    train_errs.append(mse(ytr, ridge_predict(theta, xtr)))
    val_errs.append(mse(yva, ridge_predict(theta, xva)))
train_errs, val_errs = np.array(train_errs), np.array(val_errs)

best_lam = lambdas[int(np.argmin(val_errs))]
print("best lambda (lowest val error):", best_lam)

In [ ]:
plt.plot(lambdas, train_errs, "o-", label="train error")
plt.plot(lambdas, val_errs,  "s-", label="validation error")
plt.axvline(best_lam, color="gray", ls="--", label=f"best lam = {best_lam:.1e}")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("regularization strength lambda (log scale)")
plt.ylabel("MSE (log scale)")
plt.title("Train vs validation error vs lambda (the U-curve)")
plt.legend(); plt.grid(True, which="both")
plt.show()

---
## ✍️ Exercise 1 (solution included)

Regularization works by **shrinking weights**. For the degree-12 model, compute
the L2 norm $\|\theta\|$ of the fitted weights (excluding the bias) for three
values $\lambda \in \{0, 10^{-3}, 1\}$. Confirm the norm **shrinks** as $\lambda$
grows — that *is* regularization in one number.

**Solution:**

In [ ]:
for lam in [0.0, 1e-3, 1.0]:
    theta = ridge_fit(xtr, ytr, DEGREE, lam)
    norm = np.linalg.norm(theta[1:])      # skip the bias term theta[0]
    print(f"lambda = {lam:<6}  ||theta|| = {norm:.4f}")
# The weight norm collapses as lambda increases: the penalty literally pulls the
# coefficients toward zero, which is why the fitted curve gets smoother.

## 4. L1 regularization (lasso): sparsity & feature selection

**Lasso** swaps the squared penalty for an **absolute-value** penalty:

$$J(\theta) = \frac{1}{n}\|X\theta - y\|^2 \; + \; \lambda \|\theta\|_1,
\qquad \|\theta\|_1 = \sum_j |\theta_j|.$$

The geometry of $|\cdot|$ has a remarkable effect: instead of merely *shrinking*
weights (as L2 does), L1 drives many of them to **exactly zero**. A weight of
exactly zero means that feature is **dropped** — so lasso performs automatic
**feature selection**, giving a *sparse*, interpretable model.

> **Intuition.** The gradient of $\lambda|\theta_j|$ is a constant $\pm\lambda$
> that does **not** vanish as $\theta_j \to 0$ — it keeps pushing until the weight
> hits zero and sticks. L2's gradient $2\lambda\theta_j$ fades as $\theta_j \to 0$,
> so L2 only shrinks, never zeroes out.

A full lasso solver uses **coordinate descent**; here is a short **gradient-based**
demo using a *subgradient* of $|\theta|$ (i.e. $\mathrm{sign}(\theta)$), which is
enough to see the sparsity emerge.

In [ ]:
# build a regression where ONLY a few of many features actually matter
n_samp, n_feat = 60, 10
A = rng.normal(0, 1, (n_samp, n_feat))         # 10 candidate features
true_w = np.zeros(n_feat)
true_w[2] = 3.0                                # only features 2, 5, 7 are real
true_w[5] = -2.0
true_w[7] = 1.5
target = A @ true_w + rng.normal(0, 0.1, n_samp)

def lasso_gd(A, y, lam, lr=0.01, steps=4000):
    """Minimize (1/n)||Aw - y||^2 + lam*||w||_1 by (sub)gradient descent."""
    n = A.shape[0]
    w = np.zeros(A.shape[1])
    for _ in range(steps):
        grad_mse = (2 / n) * A.T @ (A @ w - y)     # gradient of the MSE part
        grad_l1  = lam * np.sign(w)                # subgradient of lam*||w||_1
        w -= lr * (grad_mse + grad_l1)
    return w

w_lasso = lasso_gd(A, target, lam=0.3)
w_lasso[np.abs(w_lasso) < 1e-2] = 0.0             # tiny weights -> treat as zero
print("true weights :", true_w)
print("lasso weights:", np.round(w_lasso, 2))
print("nonzero count -> true:", int(np.sum(true_w != 0)),
      " lasso:", int(np.sum(w_lasso != 0)))

In [ ]:
# compare against ridge on the SAME problem: ridge shrinks but keeps everything
def ridge_gd(A, y, lam, lr=0.01, steps=4000):
    n = A.shape[0]
    w = np.zeros(A.shape[1])
    for _ in range(steps):
        grad = (2 / n) * A.T @ (A @ w - y) + 2 * lam * w   # L2 gradient
        w -= lr * grad
    return w

w_ridge = ridge_gd(A, target, lam=0.3)

idxs = np.arange(n_feat)
width = 0.4
plt.bar(idxs - width/2, w_lasso, width, label="lasso (L1)")
plt.bar(idxs + width/2, w_ridge, width, label="ridge (L2)")
plt.axhline(0, color="k", lw=0.8)
plt.xlabel("feature index"); plt.ylabel("learned weight")
plt.title("L1 zeroes out irrelevant features; L2 only shrinks them")
plt.legend(); plt.grid(True, axis="y")
plt.show()

## 5. Bias–variance, restated in terms of $\lambda$

In Chapter 9 the bias–variance trade-off was controlled by **model complexity**.
Regularization gives us a **continuous dial** for the same trade-off:

$$\text{test error} = \text{bias}^2 + \text{variance} + \text{noise}.$$

- **Small $\lambda$** → the model is free to be complex → **low bias, high
  variance** → overfitting (left/low end of the U).
- **Large $\lambda$** → weights forced toward zero → the model is rigid → **high
  bias, low variance** → underfitting (right/high end of the U).
- The **best $\lambda$** balances the two — exactly the bottom of the U-curve we
  found in Section 3.

So $\lambda$ is a smooth knob that slides the model along the bias–variance curve
*without changing its degree* — often easier to tune than discrete complexity.

---
## ✍️ Exercise 2 (solution included)

Demonstrate the **variance** half of the story. Generate **8 fresh noisy
datasets** from `true_fn`, and for each fit a degree-12 model **twice**: once
unregularized ($\lambda = 0$) and once with strong ridge ($\lambda = 0.1$).
Overlay the fitted curves. The unregularized fits should scatter wildly (high
variance); the regularized ones should cluster tightly.

**Solution:**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for trial in range(8):
    xs = np.sort(rng.uniform(0, 5, 20))
    ys = true_fn(xs) + rng.normal(0, 0.35, 20)
    for ax, lam in zip(axes, [0.0, 0.1]):
        theta = ridge_fit(xs, ys, DEGREE, lam)
        ax.plot(grid, ridge_predict(theta, grid), alpha=0.6)

for ax, lam in zip(axes, [0.0, 0.1]):
    ax.plot(grid, true_fn(grid), "k--", lw=2, label="truth")
    ax.set_ylim(-2, 5); ax.set_title(f"8 fits, lambda = {lam}")
    ax.legend(); ax.grid(True)
axes[0].set_ylabel("y")
plt.suptitle("High variance (left) vs regularized low variance (right)")
plt.tight_layout()
plt.show()
# Left: the curves fly all over the place -> high variance. Right: ridge pins
# them down near the truth -> variance reduced (at the cost of a little bias).

## 6. Weight decay: L2 inside a gradient-descent step

In Chapter 12 the gradient-descent update was

$$\theta \leftarrow \theta - \eta \, \nabla_\theta \mathrm{MSE}.$$

What happens when we add the L2 penalty $\lambda\|\theta\|^2$ to the cost? Its
gradient is $2\lambda\theta$, so the update becomes

$$\theta \leftarrow \theta - \eta\bigl(\nabla_\theta \mathrm{MSE} + 2\lambda\theta\bigr)
= \underbrace{(1 - 2\eta\lambda)\,\theta}_{\text{shrink first}} \; - \; \eta\,\nabla_\theta \mathrm{MSE}.$$

Every step **multiplies $\theta$ by a factor just below 1** *before* the usual
gradient move — it literally **decays the weights** toward zero. This is why L2
regularization is called **weight decay** in deep learning. Let's watch the
weight norm shrink during training.

In [ ]:
def gd_with_weight_decay(A, y, lam, lr=0.05, steps=200):
    """Gradient descent on MSE with L2 weight decay; track the weight norm."""
    n = A.shape[0]
    w = rng.normal(0, 1, A.shape[1])           # start from random largish weights
    norms = []
    for _ in range(steps):
        grad_mse = (2 / n) * A.T @ (A @ w - y)
        w = (1 - 2 * lr * lam) * w - lr * grad_mse   # decay, then descend
        norms.append(np.linalg.norm(w))
    return w, norms

# use the Section 4 data; compare no decay vs decay
_, norms_no   = gd_with_weight_decay(A, target, lam=0.0)
_, norms_yes  = gd_with_weight_decay(A, target, lam=0.5)

plt.plot(norms_no,  label="lambda = 0   (no weight decay)")
plt.plot(norms_yes, label="lambda = 0.5 (weight decay)")
plt.xlabel("gradient-descent step"); plt.ylabel("||theta||")
plt.title("Weight decay keeps the weight norm small during training")
plt.legend(); plt.grid(True)
plt.show()

The decay curve settles at a **smaller weight norm** — the same shrinkage ridge
achieves in closed form, now realized step-by-step inside gradient descent. The
two views (penalized cost vs. decayed update) are mathematically the same thing.

### Optional: ridge in scikit-learn

`scikit-learn`'s `Ridge` solves the very same problem. (Skipped if unavailable;
note its `alpha` plays the role of our `lambda`.)

In [ ]:
try:
    from sklearn.linear_model import Ridge
    X12 = poly_design(xtr, DEGREE)
    model = Ridge(alpha=best_lam, fit_intercept=False)   # bias already in column 0
    model.fit(X12, ytr)
    ours = ridge_fit(xtr, ytr, DEGREE, best_lam)
    print("sklearn vs our ridge weights close:",
          np.allclose(model.coef_, ours, atol=1e-6))
except Exception as e:
    print("scikit-learn not available, skipping:", e)

## Recap

- **Overfitting** shows up as wild curves and **huge weights**; regularization
  penalizes large weights so the model prefers a simpler fit.
- **L2 / ridge:** cost $J = \mathrm{MSE} + \lambda\|\theta\|^2$, solved by the
  modified normal equation $\theta = (X^\top X + \lambda I)^{-1} X^\top y$.
  Larger $\lambda$ → smaller weights → smoother fit.
- Sweeping $\lambda$ traces a **U-curve** of validation error; its bottom is the
  best regularization strength.
- **L1 / lasso:** cost uses $\|\theta\|_1$; it drives weights to **exactly zero**,
  giving sparse models and automatic **feature selection**.
- $\lambda$ is a continuous **bias–variance dial**: small = high variance
  (overfit), large = high bias (underfit).
- **Weight decay** is L2 seen inside a gradient step: each update shrinks
  $\theta$ by $(1 - 2\eta\lambda)$ before descending (connects to Chapter 12).

---
## 📝 Homework (no solutions provided)

1. **CV-tuned ridge.** Combine the two notebooks: write a function that, for a
   list of $\lambda$ values, computes the **5-fold cross-validation** MSE of the
   degree-12 ridge model and returns the best $\lambda$. Compare it to the single
   validation split's choice from Section 3.

2. **Penalizing the bias.** In `ridge_fit` we set `I[0,0] = 0` so the bias term
   is not penalized. Remove that line and re-run the $\lambda$ sweep. How does the
   fit (especially its overall level) change, and why is it usually wrong to
   shrink the intercept?

3. **Elastic net.** Combine both penalties:
   $J = \mathrm{MSE} + \lambda_1\|\theta\|_1 + \lambda_2\|\theta\|^2$. Modify
   `lasso_gd` to add the L2 gradient term, run it on the Section 4 data, and
   describe how the weights differ from pure lasso and pure ridge.

4. **Decay rate.** In `gd_with_weight_decay`, the per-step shrink factor is
   $(1 - 2\eta\lambda)$. For $\eta = 0.05$, what value of $\lambda$ would make the
   weights *grow* instead of decay (i.e. factor > 1 or < 0)? Verify experimentally
   and explain what goes wrong.